# Traditional Machine Learning Baselines (DGraphFin)

Graph Neural Networks (GNNs) have shown state-of-the-art performance in fraud detection tasks by leveraging both node features and the structural relationships (edges) between entities. 

To validate the necessity of GNNs in my pipeline, this notebook establishes the baseline performance of traditional Machine Learning algorithms, specifically **Random Forest** and **XGBoost**, on the **DGraphFin** dataset. 

DGraphFin is a large-scale financial dataset for graph anomaly detection. While the official benchmark paper ("DGraph: A Large-Scale Financial Dataset for Graph Anomaly Detection") reports AUC scores of ~0.62 to ~0.79 and AP (Average Precision) scores of ~0.02 to ~0.04 using various graph-based models (such as Node2Vec, GCN, SAGE, TGAT, etc.), traditional models and MLPs that evaluate users in isolation (ignoring the transaction graph) are expected to perform worse than structure-aware GNNs, achieving lower AUC and AP scores. This establishes a baseline and validates the thesis that graph topology is critical for identifying money laundering and fraudulent patterns.

## 1. Data Acquisition and Preprocessing

I will load the DGraphFin dataset and extract the tabular node features. In this traditional ML approach, I intentionally drop the `edge_index` (the graph structure), flattening the problem into a standard tabular classification task.

*Note: The dataset contains over 3 million nodes. I will filter out the unannotated background nodes and focus only on the binary classification of normal (0) vs. fraudulent (1) nodes.*

In [1]:
import os
import time

import numpy as np
from torch_geometric.data import Data
from torch_geometric.datasets import DGraphFin

print("Downloading and processing DGraphFin dataset...")
root: str = (
    "./data"
    if os.path.exists("./data") or os.path.exists("../notebooks/data")
    else "notebooks/data"
)
dataset: DGraphFin = DGraphFin(root=root)
data: Data = dataset[0]

print(f"Original Graph - Nodes: {data.num_nodes}, Edges: {data.num_edges}")

# Load train, validation, and test sets using official masks
X_train: np.ndarray = data.x[data.train_mask].numpy()
y_train: np.ndarray = data.y[data.train_mask].numpy()

X_val: np.ndarray = data.x[data.val_mask].numpy()
y_val: np.ndarray = data.y[data.val_mask].numpy()

X_test: np.ndarray = data.x[data.test_mask].numpy()
y_test: np.ndarray = data.y[data.test_mask].numpy()

print(
    f"Train - Samples: {X_train.shape[0]}, Normal (0): {np.sum(y_train == 0)}, Fraud (1): {np.sum(y_train == 1)}"
)
print(
    f"Val - Samples: {X_val.shape[0]}, Normal (0): {np.sum(y_val == 0)}, Fraud (1): {np.sum(y_val == 1)}"
)
print(
    f"Test - Samples: {X_test.shape[0]}, Normal (0): {np.sum(y_test == 0)}, Fraud (1): {np.sum(y_test == 1)}"
)


Original Graph - Nodes: 3700550, Edges: 4300999
Train - Samples: 857899, Normal (0): 847042, Fraud (1): 10857
Val - Samples: 183862, Normal (0): 181536, Fraud (1): 2326
Test - Samples: 183840, Normal (0): 181514, Fraud (1): 2326


## 2. Evaluation Metrics Configuration

Consistent with the benchmark paper and the GNN pipeline, I will evaluate the models using **ROC AUC (Area Under the Receiver Operating Characteristic Curve)** and **Average Precision (AP)**, alongside standard accuracy, precision, recall, and F1 scores.

In [2]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_prob: np.ndarray,
    model_name: str,
) -> dict[str, float]:
    """Computes and prints classification metrics including AUC and AP."""
    metrics: dict[str, float] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "auc": float(roc_auc_score(y_true, y_prob)),
        "ap": float(average_precision_score(y_true, y_prob)),
    }

    print(f"--- {model_name} Results ---")
    for k, v in metrics.items():
        print(f"{k.upper() if k in ['auc', 'ap'] else k.capitalize()}: {v:.4f}")

    return metrics


## 3. Random Forest Classifier

Training a standard Random Forest model on the isolated node features.

In [3]:
from sklearn.ensemble import RandomForestClassifier

print("Training Random Forest...")
start_time: float = time.time()

# Using parameters mirroring my traditional_pipeline and applying class weight balancing
rf_model: RandomForestClassifier = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, verbose=0, class_weight="balanced"
)

rf_model.fit(X_train, y_train)
rf_duration: float = time.time() - start_time
print(f"Training completed in {rf_duration:.2f} seconds.")

# Evaluation on Val set
rf_val_preds: np.ndarray = rf_model.predict(X_val)
rf_val_probs: np.ndarray = rf_model.predict_proba(X_val)[:, 1]
rf_val_metrics: dict[str, float] = compute_metrics(y_val, rf_val_preds, rf_val_probs, "Random Forest (Val)")

# Evaluation on Test set
rf_preds: np.ndarray = rf_model.predict(X_test)
rf_probs: np.ndarray = rf_model.predict_proba(X_test)[:, 1]
rf_metrics: dict[str, float] = compute_metrics(y_test, rf_preds, rf_probs, "Random Forest (Test)")


Training Random Forest...
Training completed in 64.93 seconds.
--- Random Forest (Val) Results ---
Accuracy: 0.5771
Precision: 0.0223
Recall: 0.7580
F1: 0.0434
AUC: 0.7152
AP: 0.0255
--- Random Forest (Test) Results ---
Accuracy: 0.5771
Precision: 0.0225
Recall: 0.7644
F1: 0.0437
AUC: 0.7209
AP: 0.0268


## 4. XGBoost Classifier

Training an XGBoost gradient boosting model.

In [4]:
from xgboost import XGBClassifier

print("Training XGBoost Classifier...")
start_time: float = time.time()

# Compute the positive class weight (negative count / positive count)
num_neg: int = int(np.sum(y_train == 0))
num_pos: int = int(np.sum(y_train == 1))
scale_pos_weight: float = float(num_neg / num_pos)
print(f"Calculated scale_pos_weight: {scale_pos_weight:.4f}")

xgb_model: XGBClassifier = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=42,
)

xgb_model.fit(X_train, y_train)
xgb_duration: float = time.time() - start_time
print(f"Training completed in {xgb_duration:.2f} seconds.")

# Evaluation on Val set
xgb_val_preds: np.ndarray = xgb_model.predict(X_val)
xgb_val_probs: np.ndarray = xgb_model.predict_proba(X_val)[:, 1]
xgb_val_metrics: dict[str, float] = compute_metrics(y_val, xgb_val_preds, xgb_val_probs, "XGBoost (Val)")

# Evaluation on Test set
xgb_preds: np.ndarray = xgb_model.predict(X_test)
xgb_probs: np.ndarray = xgb_model.predict_proba(X_test)[:, 1]
xgb_metrics: dict[str, float] = compute_metrics(y_test, xgb_preds, xgb_probs, "XGBoost (Test)")


Training XGBoost Classifier...
Calculated scale_pos_weight: 78.0181
Training completed in 4.68 seconds.
--- XGBoost (Val) Results ---
Accuracy: 0.5615
Precision: 0.0219
Recall: 0.7726
F1: 0.0427
AUC: 0.7166
AP: 0.0261
--- XGBoost (Test) Results ---
Accuracy: 0.5602
Precision: 0.0222
Recall: 0.7846
F1: 0.0432
AUC: 0.7233
AP: 0.0272


## 5. Conclusion: The Limitation of Tabular ML in Graph Contexts

As observed in the results above, both Random Forest and XGBoost manage to achieve a moderate AUC score of approximately 0.72 and an Average Precision (AP) of ~0.027. This closely matches the performance of the MLP baseline from the official paper (Val AUC: 0.717, Test AUC: 0.723, Val AP: 0.026, Test AP: 0.027), which also processes node features in isolation.

However, the F1-score is extremely poor (around 0.04), which is terrible. This happens because the severe class imbalance leads to an overwhelming number of false positives when the models attempt to achieve high recall, highlighting a major limitation of evaluating nodes in isolation.

Let us compare these results with the official paper's benchmarks:

| Model | Val AUC | Test AUC | Val AP | Test AP |
| :--- | :---: | :---: | :---: | :---: |
| Random Forest (Ours) | 0.7152 | 0.7209 | 0.0255 | 0.0268 |
| XGBoost (Ours) | 0.7166 | 0.7233 | 0.0261 | 0.0272 |
| MLPs | 0.7170 | 0.7230 | 0.0260 | 0.0270 |
| Node2Vec | 0.6260 | 0.6290 | 0.0190 | 0.0200 |
| DevNet | 0.7070 | 0.7150 | 0.0250 | 0.0260 |
| PC-GNN | 0.7250 | 0.7340 | 0.0290 | 0.0300 |
| CARE-GNN | 0.7340 | 0.7410 | 0.0320 | 0.0330 |
| GCN | 0.7460 | 0.7510 | 0.0350 | 0.0370 |
| AMNet | 0.7460 | 0.7520 | 0.0320 | 0.0320 |
| SAGE | 0.7700 | 0.7780 | 0.0390 | 0.0430 |
| TGAT | 0.7830 | 0.7920 | 0.0410 | 0.0440 |

### Key Observations:
* **Feature-only limits**: Our Random Forest and XGBoost baselines perform almost identically to the MLP baseline in the paper, proving that ignoring graph structure limits AUC to ~0.72 and AP to ~0.027.
* **Graph Topology is Crucial**: GNN models like SAGE and TGAT that utilize the graph structure achieve significantly higher AUC (up to 0.792) and AP (up to 0.044). This is a substantial relative increase in AP (nearly 63% improvement over tabular models), showing that network topology helps filter out false positives.
* **Relational Fraud**: The models struggle to capture the complex, underlying money-laundering rings because **fraud is inherently relational**. A node (user) might appear perfectly normal based on its 17 internal features, but its connections to known fraudulent clusters are what truly expose it.
* By stripping away the `edge_index` (the transactions and links), traditional ML models hit a hard performance ceiling with a highly skewed prediction threshold that yields a virtually unusable F1-score.

**Thesis Defense:** This empirical evidence confirms the central thesis of my project. Although the AUC score appears acceptable, the extremely low F1-score and low AP demonstrate that traditional ML algorithms are insufficient for practical fraud detection in this dataset. To improve precision and achieve both high AUC and usable F1-scores, the topology of the financial network must be processed. This necessitates the use of **Graph Neural Networks (GNNs)**, which aggregate neighbor information and structural dynamics, proving their superiority over traditional Machine Learning algorithms for relational fraud detection.